# Phase 2 — Module 3 — DAPT + Morphology-Injected XLM-RoBERTa for Sinhala Ayurvedic SBD

**This notebook is the model-training centrepiece of Module 3, Phase 2.**

### What is novel
1. **Domain-Adaptive Pretraining (DAPT)** of XLM-RoBERTa on the 70 k-line Ayurvedic recipe corpus via masked-language-modelling, *before* fine-tuning for SBD. This addresses the audit's finding that vanilla XLM-RoBERTa's modern-web Sinhala pretraining does not transfer to archaic Ayurvedic register.
2. **Morphology-injected classification head** — the 16-dim hand-crafted morphology vector (`src/morphology_features.py`) is concatenated with the contextual embedding *at the head* before the linear classifier. This is the architectural novelty.
3. **Multi-rule probabilistic weak supervision** (`src/labeling_functions.py` + `src/label_model.py`) replaces the circular single-rule labelling. Soft labels are converted into hard labels with calibrated threshold.
4. **Multi-sentence training sequences** — every training example is a concatenation of 3–8 corpus lines, so the model actually learns *mid-sequence* boundary detection rather than "STOP = end of sequence."

### What you do
Run every cell top-to-bottom. The notebook auto-detects Colab and clones the repo. **Approximate Colab T4 wall-clock:**
- DAPT (MLM): ~60 min for 3 epochs over 70 k lines
- Fine-tuning (SBD): ~25 min for 3 epochs over 12 k sequences
- Fine-tuning + morphology head: ~30 min

**Outputs** uploaded to `data/models/`:
- `dapt_xlmr/` — DAPT-checkpoint
- `sbd_xlmr_baseline/` — vanilla XLM-RoBERTa fine-tuned for SBD
- `sbd_xlmr_dapt/` — DAPT → SBD fine-tune
- `sbd_xlmr_dapt_morph/` — DAPT → SBD fine-tune with morphology-injected head

## 0 — Environment setup (Colab / local auto-detect)

In [ ]:
import os, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/madhurajayashanka/palm_leaf.git'
REPO_NAME = 'palm_leaf'

if IN_COLAB:
    if not os.path.exists(f'/content/{REPO_NAME}'):
        os.system(f'git clone {REPO_URL} /content/{REPO_NAME}')
    PROJECT_ROOT = f'/content/{REPO_NAME}'
else:
    # Resolve repo root by walking up from notebooks/phase2/
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_DIR    = os.path.join(PROJECT_ROOT, 'data')
MODELS_DIR  = os.path.join(DATA_DIR, 'models')
SRC_DIR     = os.path.join(PROJECT_ROOT, 'src')
SCRIPTS_DIR = os.path.join(PROJECT_ROOT, 'scripts')
for d in (DATA_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)
for d in (SRC_DIR, SCRIPTS_DIR):
    if d not in sys.path:
        sys.path.insert(0, d)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('IN_COLAB     =', IN_COLAB)

In [ ]:
# Hugging Face stack + tokenizer libs. Skip silently if already installed.
%pip install -q 'transformers>=4.40' 'datasets>=2.18' 'accelerate>=0.30' 'evaluate>=0.4' seqeval sentencepiece

## 1 — Build the soft training set and gold v2 (if not already present)

These scripts are deterministic and cheap; safe to re-run.

In [ ]:
if not os.path.exists(os.path.join(DATA_DIR, 'train_soft.tsv')):
    os.system(f'python {SCRIPTS_DIR}/build_soft_training_set.py --n-sequences 12000')
if not os.path.exists(os.path.join(DATA_DIR, 'gold_test_v2.tsv')):
    os.system(f'python {SCRIPTS_DIR}/build_gold_v2.py')
if not os.path.exists(os.path.join(DATA_DIR, 'safety_benchmark.jsonl')):
    os.system(f'python {SCRIPTS_DIR}/build_safety_benchmark.py')
print('Data ready.')

## 2 — Domain-Adaptive Pretraining (DAPT)

**Why:** XLM-RoBERTa's Sinhala pretraining is modern web text. Ayurvedic recipe register uses different vocabulary and morphology. We continue masked-language-modelling on `cleaned_corpus.txt` so the contextual representations specialise to this register *before* the SBD task sees any data.

Reference: Gururangan et al. ACL 2020, "Don't Stop Pretraining".

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

MODEL_NAME = 'xlm-roberta-base'
DAPT_DIR   = os.path.join(MODELS_DIR, 'dapt_xlmr')
CORPUS     = os.path.join(DATA_DIR, 'cleaned_corpus.txt')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

with open(CORPUS, encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]
print(f'{len(lines):,} lines')

# Group lines into 5-line passages for richer MLM context windows.
K = 5
passages = [' '.join(lines[i:i+K]) for i in range(0, len(lines), K)]
print(f'{len(passages):,} passages')

ds = Dataset.from_dict({'text': passages})

def tok(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)
ds_tok = ds.map(tok, batched=True, remove_columns=['text'])

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
args = TrainingArguments(
    output_dir=DAPT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_strategy='epoch',
    save_total_limit=1,
    logging_steps=200,
    report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=ds_tok, data_collator=collator)
trainer.train()
trainer.save_model(DAPT_DIR)
tokenizer.save_pretrained(DAPT_DIR)
print('DAPT checkpoint at', DAPT_DIR)

## 3 — Prepare SBD training data

Loads the multi-rule probabilistic labels from `data/train_soft.tsv` (15 k+ sequences with mid-sequence boundaries). Splits 90/10 train/val.

In [ ]:
import random
TRAIN_TSV = os.path.join(DATA_DIR, 'train_soft.tsv')

def load_conll(path):
    sequences = []
    cur_w, cur_l = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip():
                if cur_w:
                    sequences.append((cur_w, cur_l))
                cur_w, cur_l = [], []
                continue
            parts = line.split('\t')
            cur_w.append(parts[0])
            cur_l.append(1 if parts[1] == 'STOP' else 0)
    if cur_w:
        sequences.append((cur_w, cur_l))
    return sequences

all_seqs = load_conll(TRAIN_TSV)
random.Random(0).shuffle(all_seqs)
n_val = max(200, int(0.1 * len(all_seqs)))
val_seqs, train_seqs = all_seqs[:n_val], all_seqs[n_val:]
print(f'train: {len(train_seqs):,}  val: {len(val_seqs):,}')
print(f'avg tokens/seq: {sum(len(w) for w,_ in train_seqs)/len(train_seqs):.1f}')
stop_rate = sum(sum(l) for _,l in train_seqs) / sum(len(l) for _,l in train_seqs)
print(f'STOP rate (train): {stop_rate:.4f}')

## 4 — Tokenisation & label alignment

Word-level labels are assigned to the **first** sub-word of each word; subsequent sub-words get `-100` (ignored). This is the standard NER-style alignment used by Hugging Face.

In [ ]:
def encode_seqs(seqs, tok, max_len=256):
    enc = tok([w for w,_ in seqs], is_split_into_words=True,
              truncation=True, max_length=max_len, padding=False)
    labels = []
    for i, (_words, lab) in enumerate(seqs):
        wid = enc.word_ids(i)
        row = []
        prev = None
        for w in wid:
            if w is None:
                row.append(-100)
            elif w != prev:
                row.append(lab[w])
            else:
                row.append(-100)
            prev = w
        labels.append(row)
    enc['labels'] = labels
    return enc

## 5 — Variant A: baseline XLM-RoBERTa SBD (no DAPT, no morphology)

This is the *fair* reproduction of the Phase-2 prior result for comparison.

In [ ]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
from datasets import Dataset as HFDataset
import numpy as np

def train_token_clf(model_init_dir, output_dir, train_seqs, val_seqs, epochs=3, lr=2e-5):
    tok = AutoTokenizer.from_pretrained(model_init_dir)
    train_enc = encode_seqs(train_seqs, tok)
    val_enc   = encode_seqs(val_seqs,   tok)
    train_ds  = HFDataset.from_dict(train_enc)
    val_ds    = HFDataset.from_dict(val_enc)
    coll      = DataCollatorForTokenClassification(tok)
    model = AutoModelForTokenClassification.from_pretrained(
        model_init_dir, num_labels=2, id2label={0:'O',1:'STOP'}, label2id={'O':0,'STOP':1})
    args = TrainingArguments(
        output_dir=output_dir, overwrite_output_dir=True,
        num_train_epochs=epochs,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=lr, weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model='f1_stop', greater_is_better=True,
        logging_steps=100, report_to='none')
    def compute_metrics(p):
        preds = np.argmax(p.predictions, axis=-1)
        labels = p.label_ids
        mask = labels != -100
        pred = preds[mask]; gold = labels[mask]
        tp = int(((pred==1)&(gold==1)).sum())
        fp = int(((pred==1)&(gold==0)).sum())
        fn = int(((pred==0)&(gold==1)).sum())
        prec = tp/(tp+fp) if tp+fp else 0.0
        rec  = tp/(tp+fn) if tp+fn else 0.0
        f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.0
        return {'precision_stop': prec, 'recall_stop': rec, 'f1_stop': f1,
                'accuracy': float((pred==gold).mean())}
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                       tokenizer=tok, data_collator=coll, compute_metrics=compute_metrics)
    trainer.train()
    trainer.save_model(output_dir)
    tok.save_pretrained(output_dir)
    metrics = trainer.evaluate()
    print('FINAL VAL:', metrics)
    return output_dir, metrics

BASELINE_DIR = os.path.join(MODELS_DIR, 'sbd_xlmr_baseline')
_, base_metrics = train_token_clf(MODEL_NAME, BASELINE_DIR, train_seqs, val_seqs, epochs=3)

## 6 — Variant B: DAPT → SBD fine-tune

In [ ]:
DAPT_SBD_DIR = os.path.join(MODELS_DIR, 'sbd_xlmr_dapt')
_, dapt_metrics = train_token_clf(DAPT_DIR, DAPT_SBD_DIR, train_seqs, val_seqs, epochs=3)

## 7 — Variant C: DAPT → SBD fine-tune with **morphology-injected head**

**The novel architectural piece.** Instead of using `XLMRobertaForTokenClassification`'s built-in head, we define a custom head that concatenates the contextual embedding `h_i ∈ ℝ^768` with the 16-dim morphology vector `m_i ∈ ℝ^{16}` before the linear classifier:

$$\hat{y}_i = \text{softmax}(W \cdot [h_i \| m_i] + b)$$

The morphology vector encodes 13 canonical Sinhala sentence-final suffixes, broader suffix indicators, clinical verb-lemma membership, and negation markers. See `src/morphology_features.py`.

In [ ]:
import torch.nn as nn
from transformers import XLMRobertaModel, XLMRobertaConfig
from transformers.modeling_outputs import TokenClassifierOutput
from morphology_features import morph_vector, MORPH_FEATURE_DIM

class XLMRMorphForTokenClassification(nn.Module):
    """XLM-RoBERTa with [contextual ‖ morphology] token classifier head."""
    def __init__(self, init_dir, num_labels=2, morph_dim=MORPH_FEATURE_DIM, dropout=0.1):
        super().__init__()
        self.config = XLMRobertaConfig.from_pretrained(init_dir)
        self.config.num_labels = num_labels
        self.encoder = XLMRobertaModel.from_pretrained(init_dir)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size + morph_dim, num_labels)
        self.morph_dim = morph_dim
    def forward(self, input_ids=None, attention_mask=None, morph_feats=None,
                labels=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = self.dropout(out.last_hidden_state)
        if morph_feats is None:
            morph_feats = torch.zeros(h.size(0), h.size(1), self.morph_dim, device=h.device, dtype=h.dtype)
        else:
            morph_feats = morph_feats.to(h.dtype)
        joined = torch.cat([h, morph_feats], dim=-1)
        logits = self.classifier(joined)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return TokenClassifierOutput(loss=loss, logits=logits)

# Build morph_feats tensors per training example.
def encode_seqs_with_morph(seqs, tok, max_len=256):
    enc = tok([w for w,_ in seqs], is_split_into_words=True,
              truncation=True, max_length=max_len, padding=False)
    labels, morph_feats = [], []
    for i, (words, lab) in enumerate(seqs):
        wid = enc.word_ids(i)
        row_l, row_m, prev = [], [], None
        for w in wid:
            if w is None:
                row_l.append(-100); row_m.append([0.0]*MORPH_FEATURE_DIM)
            elif w != prev:
                row_l.append(lab[w]); row_m.append(morph_vector(words[w]).tolist())
            else:
                row_l.append(-100); row_m.append([0.0]*MORPH_FEATURE_DIM)
            prev = w
        labels.append(row_l); morph_feats.append(row_m)
    enc['labels'] = labels
    enc['morph_feats'] = morph_feats
    return enc

class MorphCollator:
    """Pads input_ids/attention_mask/labels and the morph_feats tensor consistently."""
    def __init__(self, tokenizer, morph_dim=MORPH_FEATURE_DIM):
        self.tokenizer = tokenizer; self.morph_dim = morph_dim
    def __call__(self, features):
        # Separate morph_feats — base collator will handle the rest.
        morphs = [f.pop('morph_feats') for f in features]
        from transformers import DataCollatorForTokenClassification
        base = DataCollatorForTokenClassification(self.tokenizer)(features)
        max_len = base['input_ids'].size(1)
        padded = []
        for m in morphs:
            if len(m) < max_len:
                m = m + [[0.0]*self.morph_dim] * (max_len - len(m))
            else:
                m = m[:max_len]
            padded.append(m)
        base['morph_feats'] = torch.tensor(padded, dtype=torch.float32)
        return base

MORPH_DIR = os.path.join(MODELS_DIR, 'sbd_xlmr_dapt_morph')
tok_m = AutoTokenizer.from_pretrained(DAPT_DIR)
train_enc_m = encode_seqs_with_morph(train_seqs, tok_m)
val_enc_m   = encode_seqs_with_morph(val_seqs,   tok_m)
train_ds_m  = HFDataset.from_dict(train_enc_m)
val_ds_m    = HFDataset.from_dict(val_enc_m)

model_m = XLMRMorphForTokenClassification(DAPT_DIR, num_labels=2)
args_m = TrainingArguments(
    output_dir=MORPH_DIR, overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='f1_stop', greater_is_better=True,
    logging_steps=100, report_to='none',
    remove_unused_columns=False)

def compute_metrics_m(p):
    preds = np.argmax(p.predictions, axis=-1)
    labels = p.label_ids
    mask = labels != -100
    pred = preds[mask]; gold = labels[mask]
    tp = int(((pred==1)&(gold==1)).sum())
    fp = int(((pred==1)&(gold==0)).sum())
    fn = int(((pred==0)&(gold==1)).sum())
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec  = tp/(tp+fn) if tp+fn else 0.0
    f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    return {'precision_stop': prec, 'recall_stop': rec, 'f1_stop': f1,
            'accuracy': float((pred==gold).mean())}

trainer_m = Trainer(model=model_m, args=args_m, train_dataset=train_ds_m, eval_dataset=val_ds_m,
                    tokenizer=tok_m, data_collator=MorphCollator(tok_m),
                    compute_metrics=compute_metrics_m)
trainer_m.train()

# Save a full snapshot (encoder + custom head) so the eval notebook can reload it.
os.makedirs(MORPH_DIR, exist_ok=True)
torch.save(model_m.state_dict(), os.path.join(MORPH_DIR, 'pytorch_model.bin'))
tok_m.save_pretrained(MORPH_DIR)
# Also persist the base config so we can rehydrate XLMRobertaConfig later.
model_m.config.save_pretrained(MORPH_DIR)
morph_metrics = trainer_m.evaluate()
print('FINAL VAL morph:', morph_metrics)

## 8 — Persist a summary JSON for the evaluation notebook

In [ ]:
import json
summary = {
    'baseline':  base_metrics,
    'dapt':      dapt_metrics,
    'dapt_morph':morph_metrics,
    'paths': {
        'baseline':  BASELINE_DIR,
        'dapt':      DAPT_SBD_DIR,
        'dapt_morph':MORPH_DIR,
        'dapt_mlm':  DAPT_DIR,
    },
}
with open(os.path.join(MODELS_DIR, 'phase2_training_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary saved.')
summary